In [1]:
!pip install pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 27.5 MB/s eta 0:00:00


In [8]:
#!/usr/bin/env python3
"""
Convert a PNG image to SVG using Potrace.

Usage:
    python png2svg.py input.png output.svg
"""

import sys
import subprocess
from pathlib import Path
from PIL import Image

def png_to_svg(input_png: str, output_svg: str, threshold: int = 128):
    """
    Convert a PNG image to SVG using potrace.

    Args:
        input_png (str): Path to the input PNG file.
        output_svg (str): Path to the output SVG file.
        threshold (int): Threshold for converting to black and white. Default is 128.
    """
    inp = Path(input_png)
    out = Path(output_svg)
    pbm = inp.with_suffix(".pbm")

    # 1) Load + convert to grayscale, then threshold to 1-bit
    img = Image.open(inp).convert("L")
    bw = img.point(lambda x: 255 if x > threshold else 0, mode="1")
    bw.save(pbm, format="PBM")

    # 2) Call potrace to generate SVG
    try:
        subprocess.run([
            "potrace",
            str(pbm),
            "--svg",
            "--output", str(out),
            "--tight",    # crop to the shape bounding box
            "--opaque",   # output only filled shapes
        ], check=True)
    finally:
        # 3) Always clean up the intermediate PBM
        pbm.unlink(missing_ok=True)

    print(f"Converted {inp} → {out} (threshold={threshold})")

if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: python png2svg.py input.png output.svg")
        sys.exit(1)
    png_to_svg(sys.argv[1], sys.argv[2])


Usage: python png2svg.py input.png output.svg


SystemExit: 1